# 第 3 天 — 对话式 AI — 也就是 Chatbot！

In [1]:
# 导入

# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
# 导入 Gradio：快速搭建可交互的 Web 演示界面（聊天框、按钮等）
import gradio as gr

In [2]:
# 从名为 .env 的文件加载环境变量
# 打印密钥前缀以便调试

load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins xxxx


In [3]:
# 初始化

# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI(api_key="xxx",base_url="http://localhost:11434/v1/")
# 选定本次实验使用的模型名称（model id）
MODEL = 'qwen3:8b'

In [11]:
# 同样，我会保持科学家模式，在实验中修改这个全局变量

system_message = "必须中文回答所有问题:You are a helpful assistant"

## 现在，编写一个新的回调

我们现在需要编写一个名为：

`chat(message, history)`

的函数，它将作为我们提供给 Gradio 的回调函数。

### 这个函数的职责

接收一条消息、先前的对话，并返回响应。


In [4]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    return "bananas"

In [5]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [6]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [7]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## 好！让我们写一个稍好一点的 chat 回调！

In [8]:

# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":msg["role"], "content":msg["content"]} for msg in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [13]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [14]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


## 好，继续前进！

用系统消息添加上下文，并给出示例回答……这又是「单次示例提示」（one shot prompting）

In [16]:
# 系统消息（system message）：聊天场景下的角色设定，等价于 system prompt
system_message = "你是一家服装店的热心助手。请温和地鼓励顾客尝试正在促销的商品。\
帽子打六折（60% off），其他多数商品打五折（50% off）。\
例如，如果顾客说「我想买一顶帽子」，\
你可以这样回答：「太好了——我们有很多帽子，其中好几款都在促销活动中。」\
如果顾客犹豫买什么，请多鼓励他们考虑帽子。"


In [17]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


In [18]:
system_message += "\n如果顾客询问鞋子，请回答今天鞋子不参与促销，\
并提醒顾客可以看看帽子！"


In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [26]:

# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
# def chat(message, history):
#     # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
#     history = [{"role":h["role"], "content":h["content"]} for h in history]
#     relevant_system_message = system_message
#     if '皮带' in message:
#         relevant_system_message += " 我们绝不售卖皮带."

#     # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
#     messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

#     # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
#     stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

#     response = ""
#     # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
#     for chunk in stream:
#         response += chunk.choices[0].delta.content or ''
#         yield response

def chat(message,history):
    new_history = []
    for msg in history:
      new_history.append({
        "role":msg["role"],
        "content":msg["content"]
      })
    history=new_history
    relevant_system_message = system_message
    if '皮带' in message:
      relevant_system_message += " 我们绝不售卖皮带."
    messages = [{"role":"system","content":relevant_system_message} ] + history + [{"role":"user","content":message}]
    stream = openai.chat.completions.create(model=MODEL,messages=messages,stream=True)
    response = ""
    for chunk in stream:
      response += chunk.choices[0].delta.content or ''
      yield response


In [27]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


In [28]:
# 思考模式：用 Gradio ChatMessage + metadata 展示「思考」折叠块
# （qwen3 / Ollama 的思考内容在 delta.reasoning，最终回答在 delta.content）

from gradio import ChatMessage
import time

def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    relevant_system_message = system_message
    if "皮带" in message:
        relevant_system_message += " 我们绝不售卖皮带."

    messages = (
        [{"role": "system", "content": relevant_system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    start = time.time()
    # 先交出一条「思考中」消息：有 title 才会显示成 Gradio 思考折叠块
    thought = ChatMessage(
        role="assistant",
        content="",
        metadata={"title": "思考过程", "status": "pending"},
    )
    yield thought

    stream = openai.chat.completions.create(
        model=MODEL, messages=messages, stream=True
    )

    thinking = ""
    answer = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        piece_think = (
            getattr(delta, "reasoning", None)
            or getattr(delta, "reasoning_content", None)
            or ""
        )
        piece_ans = delta.content or ""

        if piece_think:
            thinking += piece_think
            thought.content = thinking
            yield thought

        if piece_ans:
            answer += piece_ans
            thought.metadata["status"] = "done"
            thought.metadata["duration"] = time.time() - start
            # 思考块 + 正式回答一起 yield，界面分成两块
            yield [thought, ChatMessage(role="assistant", content=answer)]

    thought.metadata["status"] = "done"
    thought.metadata["duration"] = time.time() - start
    yield [thought, ChatMessage(role="assistant", content=answer or "(无回复)")]


In [29]:
# 启动 Gradio ChatInterface（思考模式）
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">对话式助手当然是生成式 AI 极为常见的用例，最新的前沿模型在细腻对话方面表现惊人。Gradio 也让用户界面变得容易。我们还掌握了另一项关键技能：如何用提示词提供上下文、信息和示例。
<br/><br/>
想想如何把 AI 助手应用到你的业务中，并自己做一个原型。用系统提示词给出业务上下文，并为 LLM 设定语气。</span>
        </td>
    </tr>
</table>